# Clase 1 · Notebook 02
# Tu primer modelo con Scikit-Learn: el árbol de decisión

**Introducción al Deep Learning — Módulo 1, Unidad 1: Fundamentos básicos**

Vamos a recorrer un proceso de **aprendizaje supervisado completo**, de principio a fin, usando
**Scikit-Learn**. Construiremos un **árbol de decisión** capaz de clasificar flores del famoso
conjunto de datos *Iris*.

**El flujo de trabajo que seguiremos:**

1. **Cargar** los datos.
2. **Generar** los conjuntos de entrenamiento y test (`train_test_split`).
3. **Crear** el modelo (`DecisionTreeClassifier`).
4. **Entrenar** el modelo (`fit`).
5. **Evaluar** el modelo (*accuracy* y matriz de confusión).
6. **Visualizar** el árbol resultante.

> 💡 Ejecuta las celdas en orden con **Shift + Enter**.

## 0. Conceptos previos

Recordemos la terminología de la teoría:

- **Instancia / ejemplo**: cada fila de los datos (en *Iris*, una flor).
- **Características (*features*)**: los atributos que describen cada instancia (largo/ancho de pétalo y sépalo).
- **Clase / etiqueta**: el valor que queremos predecir (la especie de la flor).
- **X**: matriz de características. **Y**: vector de etiquetas (la clase).
- **Entrenamiento**: el modelo aprende a partir de datos *con* etiqueta.
- **Inferencia**: el modelo predice la etiqueta de datos nuevos.

Un **árbol de decisión** es un algoritmo supervisado: una estructura de *raíz → nodos intermedios → hojas*.
Cada nodo interno hace una pregunta sobre una característica; cada hoja asigna una clase. Para clasificar,
se recorre un camino desde la raíz hasta una hoja. Su gran ventaja es que es **visualizable e interpretable**.

## 1. Cargar los datos

Scikit-Learn incluye varios *datasets* nativos para practicar. Usaremos **Iris**: 150 flores de 3 especies
(*setosa*, *versicolor*, *virginica*), descritas por 4 características.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
print("Características:", list(iris.feature_names))
print("Clases:", list(iris.target_names))
print("Número de instancias:", iris.data.shape[0])

# Lo miramos como tabla con Pandas para entenderlo mejor:
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["especie"] = [iris.target_names[i] for i in iris.target]
df.head()

In [ ]:
# ¿Cuántos ejemplos hay de cada especie?
df["especie"].value_counts()

## 2. Generar los conjuntos de entrenamiento y test

Separamos los datos en dos variables, como se hace habitualmente en ciencia de datos:

- **X**: las características (las 4 medidas de cada flor).
- **Y**: la clase (la especie).

Y dividimos en **entrenamiento** (para que el modelo aprenda) y **test** (para evaluarlo con datos que
*no* ha visto). Usamos `train_test_split`. El parámetro `stratify` mantiene la proporción de clases y
`random_state` hace el resultado reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

X = iris.data      # características
y = iris.target    # clase (etiqueta)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("Entrenamiento:", X_train.shape[0], "instancias")
print("Test:         ", X_test.shape[0], "instancias")

## 3. Crear el modelo

Elegimos el algoritmo: la clase `DecisionTreeClassifier`. De momento limitamos la profundidad del árbol
con `max_depth=3` para que sea **fácil de visualizar e interpretar**.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo

## 4. Entrenar el modelo

El método `fit` ejecuta el proceso de entrenamiento: el árbol aprende, a partir de `X_train` e `y_train`,
qué preguntas separan mejor las especies. `fit` es el estándar de facto para entrenar en Scikit-Learn.

In [ ]:
modelo.fit(X_train, y_train)
print("¡Modelo entrenado!")

Ya podemos usarlo para **inferencia** (predecir). Probemos con las primeras flores del conjunto de test:

In [ ]:
predicciones = modelo.predict(X_test)

print("Predicho:", [iris.target_names[i] for i in predicciones[:8]])
print("Real:    ", [iris.target_names[i] for i in y_test[:8]])

## 5. Evaluar el modelo

### 5.1 Accuracy (exactitud)

La **accuracy** es el porcentaje de aciertos sobre un conjunto de datos. La calculamos sobre el **test**,
formado por instancias que el modelo no usó para entrenar.

In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, predicciones)
print(f"Accuracy en test: {acc:.2%}")

### 5.2 Matriz de confusión

La **matriz de confusión** muestra cómo se distribuyen aciertos y errores por clase. La **diagonal principal**
contiene las clasificaciones correctas; el resto de casillas, los errores (qué clase real se confundió con
qué clase predicha).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, predicciones)
print("Matriz de confusión (filas = real, columnas = predicho):")
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusión — árbol de decisión")
plt.tight_layout()
plt.show()

También podemos pedir un **informe de clasificación** con precisión, *recall* y *F1* por clase
(estas métricas se estudian en detalle más adelante en el curso):

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, predicciones, target_names=iris.target_names))

## 6. Visualizar el árbol

Una de las grandes ventajas de los árboles de decisión es que el modelo se puede **dibujar** y entender.
Cada nodo muestra la pregunta que hace sobre una característica; siguiendo el camino llegamos a la clase predicha.

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(14, 7))
plot_tree(
    modelo,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True, rounded=True, fontsize=9, ax=ax
)
plt.title("Árbol de decisión entrenado sobre Iris")
plt.show()

## 7. Experimenta tú

Modifica el código y vuelve a ejecutar para ver cómo cambian los resultados:

1. Cambia `max_depth` (por ejemplo a `1`, `2` o `None`). ¿Cómo afecta a la *accuracy* y al tamaño del árbol?
2. Cambia `test_size` (por ejemplo a `0.2` o `0.5`). ¿Cambia el rendimiento?
3. Prueba otro algoritmo supervisado, como `KNeighborsClassifier`, y compara su *accuracy*.

> Pista para el punto 3:
> ```python
> from sklearn.neighbors import KNeighborsClassifier
> knn = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
> print(accuracy_score(y_test, knn.predict(X_test)))
> ```

## Resumen

Has completado un proceso de aprendizaje supervisado de principio a fin:
**cargar → dividir (X/Y, train/test) → crear → entrenar (`fit`) → evaluar (accuracy y matriz de confusión) → visualizar**.

Este es el esqueleto que repetiremos durante todo el curso, también cuando pasemos a las **redes de neuronas**.